In [1]:
%load_ext autoreload
%autoreload 2
# %matplotlib inline

import os
while 'notebooks' in os.getcwd():
    os.chdir("../")

import torch
from torch import nn, einsum

import quantus
import gc
import torch.nn.functional as F
import pandas as pd

from lib.helpers import plot_example_grid, short_titles_map
from lib.attributions import GradientAscentDiff, PullbackAscentDiff, \
    quantus_pullback_ascent_diff_explain_func
from lib.setup import setup_notebook
from lib.defaults import get_default_kwargs
from lib.surrogates import LayerNorm2d, PVTAttention, soften_module_inplace_
from lib.evaluator import QuantusEvaluator, default_explainers, default_metrics

c:\Users\Maciej\Documents\GitHub\SemanticPullbacks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
short_metrics_map = {
    "infidelity": "Infidelity",
    "faithfulness_correlation": "Faith.Corr",
    "monotonicity_correlation": "Mono.Corr",
    "max_sensitivity": "Max.Sens",
    "random_logit": "Rand.Logit",
}
# explainers = ["SoftPullback", "PullbackAscent", "SmoothPullback", "FusionPullback", "Gradient", "GradientAscent", "SmoothGrad", "FusionGrad", "GradientShap", "IntegratedGradients", "DeepLift", "GuidedGradCam"]
explainers = ["Gradient", "SoftPullback", "GradientAscent", "PullbackAscent", "SmoothGrad", "SmoothPullback", "FusionGrad", "FusionPullback", "IntegratedGradients", "GradientShap", "DeepLift"]

In [3]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# NumPy 2 pickles refer to numpy._core. Expose equivalent modules when this
# notebook runs with NumPy 1 so that saved results remain readable.
if int(np.__version__.split(".", maxsplit=1)[0]) < 2:
    import numpy.core as _numpy_core
    import numpy.core.multiarray as _numpy_multiarray
    import numpy.core.numeric as _numpy_numeric

    sys.modules.setdefault("numpy._core", _numpy_core)
    sys.modules.setdefault("numpy._core.multiarray", _numpy_multiarray)
    sys.modules.setdefault("numpy._core.numeric", _numpy_numeric)

def _as_float_array(value):
    if isinstance(value, (list, tuple, np.ndarray, pd.Series)):
        arr = np.asarray(value, dtype=float).reshape(-1)
    else:
        arr = np.asarray([value], dtype=float)
    return arr[np.isfinite(arr)]

def _format_val(x, precision=3):
    if np.isnan(x):
        return ""
    if abs(x) >= 1e4 or (abs(x) > 0 and abs(x) < 1e-3):
        return f"{x:.2e}"
    s = f"{x:.{precision}f}"
    return s.rstrip("0").rstrip(".") if "." in s else s

def _format_mean_std(arr, precision=3):
    if arr.size == 0:
        return ""
    mean = arr.mean()
    std = arr.std()
    mean_str = _format_val(mean, precision=precision)
    std_str = _format_val(std, precision=max(1, precision - 1))
    return f"{mean_str}±{std_str}"

def summarize_ablations_pickle(pkl_path, show=True):
    pkl_path = Path(pkl_path)
    if not pkl_path.exists():
        raise FileNotFoundError(f"File not found: {pkl_path}")

    df = pd.read_pickle(pkl_path)
    meta_cols = ["model_name", "parameter", "value", "default_value", "explainer"]
    metric_cols = [
        col
        for col in df.columns
        if col not in meta_cols and not col.startswith("is_default")
    ]
    group_cols = ["model_name", "parameter", "value", "default_value", "explainer"]

    summary_rows = []
    for _, row in df.iterrows():
        base = {col: row[col] for col in group_cols}
        for metric in metric_cols:
            arr = _as_float_array(row[metric])
            base[metric] = _format_mean_std(arr, precision=3)
        summary_rows.append(base)

    summary = (
        pd.DataFrame(summary_rows)
        .sort_values(["model_name", "parameter", "value", "explainer"])
        .reset_index(drop=True)
    )

    csv_path = pkl_path.with_name(pkl_path.stem + "_summary.csv")
    summary.to_csv(csv_path, index=False)

    print(f"Loaded: {pkl_path}")
    print(f"Saved CSV: {csv_path}")
    if show:
        display(summary)
    return summary, csv_path

def summarize_focus_pickle(focus_pkl_path, show=True):
    focus_pkl_path = Path(focus_pkl_path)
    if not focus_pkl_path.exists():
        results_dir = focus_pkl_path.parent
        available = [p.name for p in sorted(results_dir.glob("focus_model_name=*.pkl"))]
        raise FileNotFoundError(
            f"File not found: {focus_pkl_path}. Available Focus files: {available}"
        )

    focus_df = pd.read_pickle(focus_pkl_path)
    required_cols = {"explainer", "score"}
    missing_cols = required_cols.difference(focus_df.columns)
    if missing_cols:
        raise ValueError(
            f"Missing required columns in Focus results: {sorted(missing_cols)}"
        )

    focus_summary = (
        focus_df.groupby("explainer", dropna=False)["score"]
        .agg(["mean", "std", "count"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )

    focus_csv_path = focus_pkl_path.with_name(focus_pkl_path.stem + "_summary.csv")
    focus_summary.to_csv(focus_csv_path, index=False)

    print(f"Loaded: {focus_pkl_path}")
    print(f"Saved CSV: {focus_csv_path}")
    if show:
        display(focus_summary)
    return focus_summary, focus_csv_path

def _bootstrap_mean_ci(
    values,
    rng,
    n_resamples=10_000,
    confidence_level=0.95,
    chunk_size=1_000,
):
    """Return a percentile bootstrap CI for the mean of paired differences."""
    values = np.asarray(values, dtype=float).reshape(-1)
    if values.size == 0:
        return np.nan, np.nan
    if not 0 < confidence_level < 1:
        raise ValueError("confidence_level must be between 0 and 1.")
    if n_resamples < 1:
        raise ValueError("n_resamples must be at least 1.")

    bootstrap_means = np.empty(n_resamples, dtype=float)
    for start in range(0, n_resamples, chunk_size):
        stop = min(start + chunk_size, n_resamples)
        indices = rng.integers(0, values.size, size=(stop - start, values.size))
        bootstrap_means[start:stop] = values[indices].mean(axis=1)

    alpha = (1 - confidence_level) / 2
    return tuple(np.quantile(bootstrap_means, [alpha, 1 - alpha]))

def paired_differences_vs_default(
    ablations_pkl_path,
    show=True,
    explainers=("PullbackAscent", "SoftPullback"),
    bootstrap_resamples=10_000,
    bootstrap_seed=314,
):
    ablations_pkl_path = Path(ablations_pkl_path)
    if not ablations_pkl_path.exists():
        raise FileNotFoundError(f"File not found: {ablations_pkl_path}")

    df = pd.read_pickle(ablations_pkl_path)
    df = df[df["explainer"].isin(explainers)].copy()
    meta_cols = {"model_name", "parameter", "value", "default_value", "explainer", "is_default"}
    metric_cols = [col for col in df.columns if col not in meta_cols]
    rng = np.random.default_rng(bootstrap_seed)

    rows = []
    default_summary_rows = []
    for (model_name, explainer), df_expl in df.groupby(
        ["model_name", "explainer"], sort=False
    ):
        if "is_default" in df_expl.columns and df_expl["is_default"].any():
            default_rows = df_expl[df_expl["is_default"] == True]
        else:
            default_rows = df_expl[
                (df_expl["parameter"] == "default") & (df_expl["value"] == "default")
            ]

        if len(default_rows) != 1:
            raise ValueError(
                f"Expected exactly one default row for model={model_name}, "
                f"explainer={explainer}, got {len(default_rows)}"
            )

        default_row = default_rows.iloc[0]
        for metric in metric_cols:
            default_values = _as_float_array(default_row[metric])
            default_summary_rows.append({
                "model_name": model_name,
                "explainer": explainer,
                "metric": metric,
                "default_mean": float(default_values.mean()),
                "default_std": (
                    float(default_values.std(ddof=1))
                    if default_values.size > 1 else np.nan
                ),
                "count": int(default_values.size),
            })

        for _, row in df_expl.iterrows():
            is_default_row = bool(row.get("is_default", False)) or (
                row["parameter"] == "default" and row["value"] == "default"
            )
            if is_default_row:
                continue

            setting = f"{row['parameter']}={row['value']}"
            for metric in metric_cols:
                x = np.asarray(row[metric], dtype=float).reshape(-1)
                y = np.asarray(default_row[metric], dtype=float).reshape(-1)
                if x.shape != y.shape:
                    raise ValueError(
                        f"Cannot pair metric={metric}, explainer={explainer}, "
                        f"setting={setting}: variant has {x.size} values and "
                        f"default has {y.size}."
                    )
                valid = np.isfinite(x) & np.isfinite(y)
                x = x[valid]
                y = y[valid]
                n = len(x)
                if n == 0:
                    continue

                diff = x - y
                ci95_low, ci95_high = _bootstrap_mean_ci(
                    diff,
                    rng=rng,
                    n_resamples=bootstrap_resamples,
                )
                rows.append(
                    {
                        "model_name": row["model_name"],
                        "explainer": explainer,
                        "setting": setting,
                        "parameter": row["parameter"],
                        "value": row["value"],
                        "default_value": row["default_value"],
                        "metric": metric,
                        "default_mean": float(np.mean(y)),
                        "variant_mean": float(np.mean(x)),
                        "delta_mean": float(np.mean(diff)),
                        "delta_std": float(np.std(diff, ddof=1)) if n > 1 else np.nan,
                        "ci95_low": float(ci95_low),
                        "ci95_high": float(ci95_high),
                        "paired_count": int(n),
                    }
                )

    paired_df = pd.DataFrame(rows).sort_values(
        ["explainer", "parameter", "value", "metric"]
    ).reset_index(drop=True)

    paired_csv_path = ablations_pkl_path.with_name(
        ablations_pkl_path.stem + "_paired_diff_vs_default.csv"
    )
    paired_df.to_csv(paired_csv_path, index=False)

    metric_directions = {
        "infidelity": "↓",
        "faithfulness_correlation": "↑",
        "faithfulness_estimate": "↑",
        "monotonicity_correlation": "↑",
        "max_sensitivity": "↓",
        "random_logit": "↓",
    }
    metric_labels = {
        metric: f"{short_metrics_map.get(metric, metric)} {metric_directions.get(metric, '')}".rstrip()
        for metric in metric_cols
    }

    paired_display = paired_df.copy()
    paired_display["metric"] = paired_display["metric"].map(metric_labels)
    paired_display["delta_ci95"] = paired_display.apply(
        lambda r: (
            f"{r['delta_mean']:.4f} "
            f"[{r['ci95_low']:.4f}, {r['ci95_high']:.4f}]"
        ),
        axis=1,
    )
    paired_pivot = (
        paired_display.assign(
            value_num=pd.to_numeric(paired_display["value"], errors="coerce")
        )
        .sort_values(["explainer", "parameter", "value_num", "value"])
        .pivot_table(
            index=["explainer", "parameter", "default_value", "value"],
            columns="metric",
            values="delta_ci95",
            aggfunc="first",
        )
        .reset_index()
    )

    default_summary = pd.DataFrame(default_summary_rows)
    default_summary["metric"] = default_summary["metric"].map(metric_labels)
    default_summary["mean_std"] = default_summary.apply(
        lambda r: f"{r['default_mean']:.4f}±{r['default_std']:.4f}",
        axis=1,
    )
    default_pivot = default_summary.pivot(
        index=["model_name", "explainer"],
        columns="metric",
        values="mean_std",
    ).reset_index()
    default_pivot.columns.name = None

    print(f"Loaded: {ablations_pkl_path}")
    print(f"Saved paired difference CSV: {paired_csv_path}")
    print("Default scores (mean±std):")
    if show:
        display(default_pivot)
    print(
        f"Paired delta format: variant - default, mean [95% percentile bootstrap CI] "
        f"using {bootstrap_resamples:,} resamples (seed={bootstrap_seed})."
    )
    print(f"Paired counts present in the detailed CSV: {sorted(paired_df['paired_count'].unique())}")
    if show:
        display(paired_pivot)
    return paired_df, paired_pivot, paired_csv_path

EXPLAINER_PAIRS = (
    ("SoftPullback", "Gradient", "SoftPullback - Gradient"),
    ("PullbackAscent", "GradientAscent", "PullbackAscent - GradientAscent"),
    ("SmoothPullback", "SmoothGrad", "SmoothPullback - SmoothGrad"),
    ("FusionPullback", "FusionGrad", "FusionPullback - FusionGrad"),
)

def paired_explainer_differences(
    results_pkl_path,
    metric_overrides=None,
    pairs=EXPLAINER_PAIRS,
    show=True,
    bootstrap_resamples=10_000,
    bootstrap_seed=314,
):
    """Compute per-image Pullback-minus-Gradient contrasts within each run."""
    results_pkl_path = Path(results_pkl_path)
    if not results_pkl_path.exists():
        raise FileNotFoundError(f"File not found: {results_pkl_path}")

    base_df = pd.read_pickle(results_pkl_path)
    metric_frames = {metric: base_df for metric in base_df.columns}
    metric_sources = {metric: results_pkl_path for metric in base_df.columns}

    for metric, override_path in (metric_overrides or {}).items():
        override_path = Path(override_path)
        if not override_path.exists():
            raise FileNotFoundError(f"File not found: {override_path}")
        override_df = pd.read_pickle(override_path)
        if metric not in override_df.columns:
            raise ValueError(f"Metric {metric!r} is missing from {override_path}.")
        metric_frames[metric] = override_df
        metric_sources[metric] = override_path

    rng = np.random.default_rng(bootstrap_seed)
    rows = []
    for pullback_name, gradient_name, pair_label in pairs:
        for metric, metric_df in metric_frames.items():
            missing = {pullback_name, gradient_name}.difference(metric_df.index)
            if missing:
                raise ValueError(
                    f"Missing explainers {sorted(missing)} for metric={metric!r} "
                    f"in {metric_sources[metric]}."
                )

            pullback_values = np.asarray(
                metric_df.at[pullback_name, metric], dtype=float
            ).reshape(-1)
            gradient_values = np.asarray(
                metric_df.at[gradient_name, metric], dtype=float
            ).reshape(-1)
            if pullback_values.shape != gradient_values.shape:
                raise ValueError(
                    f"Cannot pair {pair_label}, metric={metric!r}: Pullback has "
                    f"{pullback_values.size} values and Gradient has "
                    f"{gradient_values.size}."
                )

            valid = np.isfinite(pullback_values) & np.isfinite(gradient_values)
            paired_pullback = pullback_values[valid]
            paired_gradient = gradient_values[valid]
            differences = paired_pullback - paired_gradient
            if differences.size == 0:
                continue

            ci95_low, ci95_high = _bootstrap_mean_ci(
                differences,
                rng=rng,
                n_resamples=bootstrap_resamples,
            )
            rows.append({
                "result_set": results_pkl_path.stem,
                "pair": pair_label,
                "pullback_explainer": pullback_name,
                "gradient_explainer": gradient_name,
                "metric": metric,
                "metric_source": metric_sources[metric].name,
                "pullback_mean": float(paired_pullback.mean()),
                "gradient_mean": float(paired_gradient.mean()),
                "delta_mean": float(differences.mean()),
                "delta_std": (
                    float(differences.std(ddof=1))
                    if differences.size > 1 else np.nan
                ),
                "ci95_low": float(ci95_low),
                "ci95_high": float(ci95_high),
                "paired_count": int(differences.size),
            })

    paired_df = pd.DataFrame(rows).sort_values(["pair", "metric"]).reset_index(drop=True)
    paired_csv_path = results_pkl_path.with_name(
        results_pkl_path.stem + "_paired_explainers.csv"
    )
    paired_df.to_csv(paired_csv_path, index=False)

    metric_directions = {
        "infidelity": "↓",
        "faithfulness_correlation": "↑",
        "faithfulness_estimate": "↑",
        "monotonicity_correlation": "↑",
        "max_sensitivity": "↓",
        "random_logit": "↓",
    }
    metric_labels = {
        metric: f"{short_metrics_map.get(metric, metric)} {metric_directions.get(metric, '')}".rstrip()
        for metric in paired_df["metric"].unique()
    }
    paired_display = paired_df.copy()
    paired_display["metric"] = paired_display["metric"].map(metric_labels)
    paired_display["delta_ci95"] = paired_display.apply(
        lambda row: (
            f"{row['delta_mean']:.4f} "
            f"[{row['ci95_low']:.4f}, {row['ci95_high']:.4f}]"
        ),
        axis=1,
    )
    paired_pivot = paired_display.pivot(
        index="pair",
        columns="metric",
        values="delta_ci95",
    ).reset_index()
    paired_pivot.columns.name = None

    print(f"Loaded base results: {results_pkl_path}")
    for metric, source_path in metric_sources.items():
        print(f"Metric source [{metric}]: {source_path}")
    print(f"Saved paired explainer CSV: {paired_csv_path}")
    print(
        f"Delta format: Pullback - Gradient, mean [95% percentile bootstrap CI] "
        f"using {bootstrap_resamples:,} resamples (seed={bootstrap_seed})."
    )
    print(f"Paired counts: {sorted(paired_df['paired_count'].unique())}")
    if show:
        display(paired_pivot)
    return paired_df, paired_pivot, paired_csv_path

In [4]:
def paired_focus_explainer_differences(
    results_pkl_path,
    pairs=EXPLAINER_PAIRS,
    show=True,
    bootstrap_resamples=10_000,
    bootstrap_seed=314,
):
    """Compute paired Pullback-minus-Gradient Focus score contrasts."""
    results_pkl_path = Path(results_pkl_path)
    if not results_pkl_path.exists():
        raise FileNotFoundError(f"File not found: {results_pkl_path}")

    focus_df = pd.read_pickle(results_pkl_path)
    required_cols = {"explainer", "score", "evaluation_batch", "mosaic_index"}
    missing_cols = required_cols.difference(focus_df.columns)
    if missing_cols:
        raise ValueError(
            f"Missing required Focus columns: {sorted(missing_cols)}"
        )

    sample_keys = ["evaluation_batch", "mosaic_index"]
    duplicate_mask = focus_df.duplicated(sample_keys + ["explainer"], keep=False)
    if duplicate_mask.any():
        raise ValueError(
            "Focus results contain duplicate rows for an explainer and sample key."
        )

    score_matrix = focus_df.pivot(
        index=sample_keys,
        columns="explainer",
        values="score",
    ).sort_index()

    rng = np.random.default_rng(bootstrap_seed)
    rows = []
    for pullback_name, gradient_name, pair_label in pairs:
        missing_explainers = {pullback_name, gradient_name}.difference(score_matrix.columns)
        if missing_explainers:
            raise ValueError(
                f"Missing explainers {sorted(missing_explainers)} in {results_pkl_path}."
            )

        pullback_values = pd.to_numeric(
            score_matrix[pullback_name], errors="coerce"
        ).to_numpy(dtype=float)
        gradient_values = pd.to_numeric(
            score_matrix[gradient_name], errors="coerce"
        ).to_numpy(dtype=float)
        valid = np.isfinite(pullback_values) & np.isfinite(gradient_values)
        paired_pullback = pullback_values[valid]
        paired_gradient = gradient_values[valid]
        differences = paired_pullback - paired_gradient
        if differences.size == 0:
            continue

        ci95_low, ci95_high = _bootstrap_mean_ci(
            differences,
            rng=rng,
            n_resamples=bootstrap_resamples,
        )
        rows.append({
            "result_set": results_pkl_path.stem,
            "pair": pair_label,
            "pullback_explainer": pullback_name,
            "gradient_explainer": gradient_name,
            "metric": "score",
            "metric_source": results_pkl_path.name,
            "pullback_mean": float(paired_pullback.mean()),
            "gradient_mean": float(paired_gradient.mean()),
            "delta_mean": float(differences.mean()),
            "delta_std": (
                float(differences.std(ddof=1))
                if differences.size > 1 else np.nan
            ),
            "ci95_low": float(ci95_low),
            "ci95_high": float(ci95_high),
            "paired_count": int(differences.size),
        })

    paired_df = pd.DataFrame(rows).sort_values(
        ["pair", "metric"]
    ).reset_index(drop=True)
    paired_csv_path = results_pkl_path.with_name(
        results_pkl_path.stem + "_paired_explainers.csv"
    )
    paired_df.to_csv(paired_csv_path, index=False)

    paired_display = paired_df.copy()
    paired_display["metric"] = "Focus score ↑"
    paired_display["delta_ci95"] = paired_display.apply(
        lambda row: (
            f"{row['delta_mean']:.4f} "
            f"[{row['ci95_low']:.4f}, {row['ci95_high']:.4f}]"
        ),
        axis=1,
    )
    paired_pivot = paired_display.pivot(
        index="pair",
        columns="metric",
        values="delta_ci95",
    ).reset_index()
    paired_pivot.columns.name = None

    print(f"Loaded Focus results: {results_pkl_path}")
    print(f"Saved paired explainer CSV: {paired_csv_path}")
    print(
        f"Delta format: Pullback - Gradient, mean [95% percentile bootstrap CI] "
        f"using {bootstrap_resamples:,} resamples (seed={bootstrap_seed})."
    )
    print(f"Paired counts: {sorted(paired_df['paired_count'].unique())}")
    if show:
        display(paired_pivot)
    return paired_df, paired_pivot, paired_csv_path

In [9]:
def merge_with_priority(df_big, df_small, key_col="Faith.Corr"):
    # we merge results as we re-genenerated the faithfulness_correlation metric for different set of patches
    # the previous selection had too large patches, which inflated the correlation score
    result = df_big.loc[df_small.index].copy()
    result[key_col] = df_small[key_col]
    return result

def process_df(
    df_path,
    precision=2,
    uncertainty="std",
    confidence_level=0.95,
    bootstrap_resamples=10_000,
    bootstrap_seed=314,
):
    """Use uncertainty='bootstrap_ci' to report mean [95% CI] instead of mean±std."""
    selected_columns = list(short_metrics_map.keys())
    
    df = QuantusEvaluator.load_results(df_path)
    df = df[
        [col for col in selected_columns if col in df.columns]
    ].rename(columns=short_metrics_map)
    df = df.loc[[idx for idx in explainers if idx in df.index]]
    df = df.rename(index=short_titles_map)
    
    print("num_samples:", len(df.iloc[0].iloc[0]))
    
    df = QuantusEvaluator.summarize_results(
        df,
        precision=precision,
        uncertainty=uncertainty,
        confidence_level=confidence_level,
        bootstrap_resamples=bootstrap_resamples,
        bootstrap_seed=bootstrap_seed,
    )
    return df

def add_focus_to_df(df, focus_pkl_path):
    focus_df = pd.read_pickle(focus_pkl_path)
    focus_summary = focus_df.groupby("explainer")["score"].apply(
        lambda scores: (
            f"{scores.mean():.2f}±{scores.std():.2f}"
            if len(scores) > 1 else f"{scores.mean():.2f}±nan"
        )
    )
    focus_summary.index = focus_summary.index.map(short_titles_map)

    result = df.drop(columns="Focus", errors="ignore").copy()
    result.insert(1, "Focus", focus_summary.reindex(result.index))
    return result

In [10]:
vgg_df = process_df("results/quantus_vgg_fc_nips_20_50_vgg11_bn")
vgg_df = add_focus_to_df(
    vgg_df,
    "results/focus/quantus_vgg_focus_1000_model_name=vgg11_bn.pkl",
)
vgg_df.to_csv('results/vgg_df_20_50.csv', index=True)
vgg_df

num_samples: 1000


,Infidelity,Focus,Faith.Corr,Mono.Corr,Max.Sens,Rand.Logit
Grad,100.09±107.94,0.54±0.13,0.46±0.35,0.32±0.37,0.90±0.14,-0.05±0.29
SfP,16.30±31.68,0.61±0.11,0.61±0.30,0.37±0.39,0.21±0.08,-0.06±0.33
GA,18.52±35.24,0.56±0.10,0.51±0.30,0.23±0.32,1.22±0.06,6.65e-04±0.07
PA,5.62±13.18,0.62±0.11,0.57±0.29,0.31±0.37,0.39±0.11,0.14±0.13
SmoothG,52.04±70.80,0.56±0.11,0.51±0.34,0.38±0.36,0.78±0.11,-0.02±0.19
SmoothPb,14.03±30.57,0.62±0.12,0.55±0.32,0.31±0.40,0.40±0.11,-0.05±0.32
FusionG,38.65±60.23,0.56±0.11,0.51±0.34,0.36±0.34,0.95±0.13,-0.01±0.16
FusionPb,16.06±34.46,0.61±0.12,0.51±0.33,0.31±0.39,0.51±0.11,-0.04±0.29
IG,75.80±92.29,0.55±0.16,0.43±0.36,0.38±0.41,0.75±0.15,-0.05±0.27
GradShap,76.05±94.76,0.54±0.16,0.43±0.36,0.38±0.41,1.09±0.22,-0.04±0.26


In [11]:
pvt_df_fc = process_df("results/quantus_pvt_fc_nips_20_50_pvt_v2_b1")
pvt_df = process_df("results/quantus_pvt_it_nips_20_50_pvt_v2_b1")
pvt_merged_df = merge_with_priority(pvt_df, pvt_df_fc)
pvt_merged_df = add_focus_to_df(
    pvt_merged_df,
    "results/focus/quantus_pvt_focus_model_name=pvt_v2_b1.pkl",
)
pvt_merged_df.to_csv('results/pvt_df_20_50.csv', index=True)
pvt_merged_df

num_samples: 1000
num_samples: 1000


,Infidelity,Focus,Faith.Corr,Mono.Corr,Max.Sens,Rand.Logit
Grad,8.91±7.89,0.56±0.18,0.11±0.33,0.19±0.43,1.03±0.16,-0.03±0.30
SfP,6.26±6.40,0.62±0.18,0.12±0.34,0.16±0.43,1.07±0.18,-0.01±0.35
GA,4.51±4.07,0.55±0.10,0.12±0.33,0.16±0.35,1.24±0.07,0.01±0.06
PA,1.63±1.03,0.55±0.10,0.12±0.36,0.22±0.33,0.85±0.12,0.12±0.11
SmoothG,8.80±7.08,0.59±0.11,0.10±0.38,0.28±0.38,0.57±0.10,-0.01±0.13
SmoothPb,4.97±5.46,0.64±0.10,0.10±0.39,0.23±0.38,0.52±0.09,0.01±0.23
FusionG,6.67±5.60,0.61±0.10,0.10±0.39,0.26±0.38,0.98±0.16,5.73e-04±0.11
FusionPb,5.16±5.72,0.61±0.11,0.09±0.39,0.20±0.38,0.75±0.11,0.01±0.19
IG,12.58±8.39,0.57±0.19,0.07±0.32,0.17±0.45,1.37±0.37,0.01±0.33
GradShap,12.43±8.45,0.56±0.19,0.07±0.32,0.17±0.45,2.47±1.56,0.00±0.34


In [12]:
resnet_df_fc = process_df("results/quantus_resnet_fc_nips_20_50_resnet50")
resnet_df = process_df("results/quantus_resnet_it_nips_20_50_resnet50")
resnet_merged_df = merge_with_priority(resnet_df, resnet_df_fc)
resnet_merged_df = add_focus_to_df(
    resnet_merged_df,
    "results/focus/quantus_resnet_focus_1000_model_name=resnet50.pkl",
)
resnet_merged_df.to_csv('results/resnet_df_20_50.csv', index=True)
resnet_merged_df

num_samples: 1000
num_samples: 1000


,Infidelity,Focus,Faith.Corr,Mono.Corr,Max.Sens,Rand.Logit
Grad,83.29±68.15,0.63±0.14,0.24±0.36,0.28±0.39,0.95±0.17,-0.03±0.23
SfP,5.99±5.67,0.75±0.13,0.39±0.37,0.42±0.39,0.12±0.04,-0.06±0.42
GA,29.48±35.36,0.58±0.10,0.28±0.37,0.23±0.32,1.27±0.04,0.00±0.05
PA,5.38±4.83,0.73±0.13,0.38±0.37,0.42±0.40,0.24±0.09,0.21±0.19
SmoothG,66.88±58.16,0.64±0.12,0.33±0.40,0.36±0.36,0.64±0.08,-0.02±0.20
SmoothPb,8.12±10.13,0.71±0.16,0.33±0.39,0.35±0.40,0.22±0.05,-0.06±0.40
FusionG,58.51±57.80,0.63±0.12,0.34±0.39,0.36±0.35,0.86±0.09,-0.01±0.17
FusionPb,10.13±14.55,0.69±0.16,0.31±0.40,0.35±0.40,0.36±0.08,-0.05±0.39
IG,66.83±61.21,0.63±0.17,0.24±0.38,0.35±0.41,0.79±0.21,-0.03±0.23
GradShap,69.07±62.93,0.62±0.17,0.23±0.38,0.34±0.41,1.40±0.41,-0.02±0.24


## Paired Pullback–Gradient explainer comparisons

Each contrast is computed within one `QuantusEvaluator` result file. `evaluate_loader` evaluates every explainer on the same batch before advancing the loader, so positions in the score arrays refer to the same images. For ResNet and PVT, the corrected Faithfulness Correlation is taken from the corresponding `*_fc_*` run, and both members of a pair are always read from that same run. Entries report `Pullback - Gradient` as a mean paired difference with a 95% percentile bootstrap confidence interval.

In [26]:
paired_focus_results = {
    path.stem: paired_focus_explainer_differences(path)
    for path in sorted(Path("results/focus").glob("*.pkl"))
}
if not paired_focus_results:
    raise FileNotFoundError("No Focus pickle files found in results/focus/.")

Loaded Focus results: results\focus\quantus_pvt_focus_model_name=pvt_v2_b1.pkl
Saved paired explainer CSV: results\focus\quantus_pvt_focus_model_name=pvt_v2_b1_paired_explainers.csv
Delta format: Pullback - Gradient, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts: [np.int64(1000)]


,pair,Focus score ↑
0,FusionPullback - FusionGrad,"0.0084 [0.0045, 0.0123]"
1,PullbackAscent - GradientAscent,"0.0036 [0.0003, 0.0069]"
2,SmoothPullback - SmoothGrad,"0.0411 [0.0363, 0.0459]"
3,SoftPullback - Gradient,"0.0599 [0.0541, 0.0655]"


Loaded Focus results: results\focus\quantus_resnet_focus_1000_model_name=resnet50.pkl
Saved paired explainer CSV: results\focus\quantus_resnet_focus_1000_model_name=resnet50_paired_explainers.csv
Delta format: Pullback - Gradient, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts: [np.int64(1000)]


,pair,Focus score ↑
0,FusionPullback - FusionGrad,"0.0572 [0.0509, 0.0637]"
1,PullbackAscent - GradientAscent,"0.1419 [0.1353, 0.1487]"
2,SmoothPullback - SmoothGrad,"0.0733 [0.0664, 0.0802]"
3,SoftPullback - Gradient,"0.1116 [0.1023, 0.1208]"


Loaded Focus results: results\focus\quantus_vgg_focus_1000_model_name=vgg11_bn.pkl
Saved paired explainer CSV: results\focus\quantus_vgg_focus_1000_model_name=vgg11_bn_paired_explainers.csv
Delta format: Pullback - Gradient, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts: [np.int64(900)]


,pair,Focus score ↑
0,FusionPullback - FusionGrad,"0.0530 [0.0490, 0.0570]"
1,PullbackAscent - GradientAscent,"0.0666 [0.0619, 0.0712]"
2,SmoothPullback - SmoothGrad,"0.0630 [0.0588, 0.0674]"
3,SoftPullback - Gradient,"0.0702 [0.0636, 0.0768]"


In [27]:
paired_explainers_vgg, paired_explainers_vgg_table, paired_explainers_vgg_csv = (
    paired_explainer_differences(
        "results/quantus_vgg_fc_nips_20_50_vgg11_bn.pkl"
    )
)

Loaded base results: results\quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Metric source [infidelity]: results\quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Metric source [faithfulness_correlation]: results\quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Metric source [faithfulness_estimate]: results\quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Metric source [monotonicity_correlation]: results\quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Metric source [max_sensitivity]: results\quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Metric source [random_logit]: results\quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Saved paired explainer CSV: results\quantus_vgg_fc_nips_20_50_vgg11_bn_paired_explainers.csv
Delta format: Pullback - Gradient, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts: [np.int64(1000)]


,pair,Faith.Corr ↑,Infidelity ↓,Max.Sens ↓,Mono.Corr ↑,Rand.Logit ↓,faithfulness_estimate ↑
0,FusionPullback - FusionGrad,"0.0070 [-0.0031, 0.0170]","-22.5950 [-26.1671, -19.2468]","-0.4421 [-0.4479, -0.4363]","-0.0483 [-0.0627, -0.0338]","-0.0224 [-0.0346, -0.0101]","-0.1152 [-0.1330, -0.0974]"
1,PullbackAscent - GradientAscent,"0.0618 [0.0505, 0.0734]","-12.9064 [-15.1549, -10.7841]","-0.8275 [-0.8320, -0.8230]","0.0778 [0.0566, 0.0995]","0.1354 [0.1304, 0.1404]","0.1115 [0.0842, 0.1392]"
2,SmoothPullback - SmoothGrad,"0.0322 [0.0216, 0.0431]","-38.0143 [-42.3591, -33.8440]","-0.3831 [-0.3867, -0.3794]","-0.0752 [-0.0906, -0.0599]","-0.0237 [-0.0376, -0.0099]","-0.0801 [-0.0977, -0.0625]"
3,SoftPullback - Gradient,"0.1489 [0.1319, 0.1657]","-83.7863 [-90.4122, -77.2206]","-0.6860 [-0.6922, -0.6799]","0.0589 [0.0364, 0.0815]","-0.0101 [-0.0207, 0.0006]","0.0641 [0.0377, 0.0907]"


In [28]:
paired_explainers_resnet, paired_explainers_resnet_table, paired_explainers_resnet_csv = (
    paired_explainer_differences(
        "results/quantus_resnet_it_nips_20_50_resnet50.pkl",
        metric_overrides={
            "faithfulness_correlation": (
                "results/quantus_resnet_fc_nips_20_50_resnet50.pkl"
            ),
        },
    )
)

Loaded base results: results\quantus_resnet_it_nips_20_50_resnet50.pkl
Metric source [infidelity]: results\quantus_resnet_it_nips_20_50_resnet50.pkl
Metric source [faithfulness_correlation]: results\quantus_resnet_fc_nips_20_50_resnet50.pkl
Metric source [faithfulness_estimate]: results\quantus_resnet_it_nips_20_50_resnet50.pkl
Metric source [monotonicity_correlation]: results\quantus_resnet_it_nips_20_50_resnet50.pkl
Metric source [max_sensitivity]: results\quantus_resnet_it_nips_20_50_resnet50.pkl
Metric source [random_logit]: results\quantus_resnet_it_nips_20_50_resnet50.pkl
Saved paired explainer CSV: results\quantus_resnet_it_nips_20_50_resnet50_paired_explainers.csv
Delta format: Pullback - Gradient, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts: [np.int64(1000)]


,pair,Faith.Corr ↑,Infidelity ↓,Max.Sens ↓,Mono.Corr ↑,Rand.Logit ↓,faithfulness_estimate ↑
0,FusionPullback - FusionGrad,"-0.0385 [-0.0537, -0.0233]","-48.3830 [-51.9253, -44.9511]","-0.4932 [-0.4997, -0.4867]","-0.0073 [-0.0252, 0.0100]","-0.0372 [-0.0587, -0.0158]","-0.0294 [-0.0532, -0.0056]"
1,PullbackAscent - GradientAscent,"0.1040 [0.0874, 0.1208]","-24.0957 [-26.3416, -21.9837]","-1.0230 [-1.0278, -1.0182]","0.1903 [0.1655, 0.2149]","0.2105 [0.1999, 0.2210]","0.2631 [0.2339, 0.2916]"
2,SmoothPullback - SmoothGrad,"-0.0058 [-0.0217, 0.0094]","-58.7615 [-62.4075, -55.3068]","-0.4187 [-0.4232, -0.4142]","-0.0128 [-0.0314, 0.0053]","-0.0405 [-0.0623, -0.0187]","-0.0008 [-0.0259, 0.0239]"
3,SoftPullback - Gradient,"0.1472 [0.1226, 0.1710]","-77.3050 [-81.3633, -73.2595]","-0.8335 [-0.8434, -0.8237]","0.1444 [0.1164, 0.1721]","-0.0291 [-0.0509, -0.0067]","0.1475 [0.1170, 0.1783]"


In [29]:
paired_explainers_pvt, paired_explainers_pvt_table, paired_explainers_pvt_csv = (
    paired_explainer_differences(
        "results/quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl",
        metric_overrides={
            "faithfulness_correlation": (
                "results/quantus_pvt_fc_nips_20_50_pvt_v2_b1.pkl"
            ),
        },
    )
)

Loaded base results: results\quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl
Metric source [infidelity]: results\quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl
Metric source [faithfulness_correlation]: results\quantus_pvt_fc_nips_20_50_pvt_v2_b1.pkl
Metric source [faithfulness_estimate]: results\quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl
Metric source [monotonicity_correlation]: results\quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl
Metric source [max_sensitivity]: results\quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl
Metric source [random_logit]: results\quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl
Saved paired explainer CSV: results\quantus_pvt_it_nips_20_50_pvt_v2_b1_paired_explainers.csv
Delta format: Pullback - Gradient, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts: [np.int64(1000)]


,pair,Faith.Corr ↑,Infidelity ↓,Max.Sens ↓,Mono.Corr ↑,Rand.Logit ↓,faithfulness_estimate ↑
0,FusionPullback - FusionGrad,"-0.0088 [-0.0178, 0.0002]","-1.5167 [-1.9709, -1.0603]","-0.2258 [-0.2365, -0.2151]","-0.0617 [-0.0774, -0.0462]","0.0061 [-0.0052, 0.0177]","-0.0296 [-0.0486, -0.0106]"
1,PullbackAscent - GradientAscent,"0.0044 [-0.0109, 0.0199]","-2.8716 [-3.1287, -2.6310]","-0.3875 [-0.3924, -0.3824]","0.0544 [0.0365, 0.0730]","0.1158 [0.1088, 0.1229]","0.1415 [0.1196, 0.1642]"
2,SmoothPullback - SmoothGrad,"0.0059 [-0.0050, 0.0167]","-3.8244 [-4.3190, -3.3446]","-0.0497 [-0.0540, -0.0454]","-0.0483 [-0.0641, -0.0324]","0.0200 [0.0055, 0.0345]","-0.0255 [-0.0461, -0.0050]"
3,SoftPullback - Gradient,"0.0137 [0.0005, 0.0270]","-2.6502 [-3.1834, -2.1285]","0.0317 [0.0190, 0.0439]","-0.0208 [-0.0394, -0.0023]","0.0232 [-0.0030, 0.0496]","0.0426 [0.0245, 0.0609]"


In [18]:
def merge_focus_into_paired_results(paired_df, paired_table, focus_df, csv_path):
    focus_values = focus_df[["pair", "delta_mean", "ci95_low", "ci95_high"]].copy()
    focus_values = focus_values.rename(columns={"delta_mean": "Focus"})
    paired_df = paired_df.drop(columns="Focus", errors="ignore").merge(
        focus_values[["pair", "Focus"]],
        on="pair",
        how="left",
        validate="many_to_one",
    )

    focus_values["Focus"] = focus_values.apply(
        lambda row: (
            f"{row['Focus']:.4f} "
            f"[{row['ci95_low']:.4f}, {row['ci95_high']:.4f}]"
        ),
        axis=1,
    )
    paired_table = paired_table.drop(columns="Focus", errors="ignore").merge(
        focus_values[["pair", "Focus"]],
        on="pair",
        how="left",
        validate="one_to_one",
    )

    paired_df.to_csv(csv_path, index=False)
    return paired_df, paired_table

paired_explainers_vgg, paired_explainers_vgg_table = merge_focus_into_paired_results(
    paired_explainers_vgg,
    paired_explainers_vgg_table,
    paired_focus_results["quantus_vgg_focus_1000_model_name=vgg11_bn"][0],
    paired_explainers_vgg_csv,
)
paired_explainers_resnet, paired_explainers_resnet_table = merge_focus_into_paired_results(
    paired_explainers_resnet,
    paired_explainers_resnet_table,
    paired_focus_results["quantus_resnet_focus_1000_model_name=resnet50"][0],
    paired_explainers_resnet_csv,
)
paired_explainers_pvt, paired_explainers_pvt_table = merge_focus_into_paired_results(
    paired_explainers_pvt,
    paired_explainers_pvt_table,
    paired_focus_results["quantus_pvt_focus_model_name=pvt_v2_b1"][0],
    paired_explainers_pvt_csv,
)

display(paired_explainers_vgg_table)
display(paired_explainers_resnet_table)
display(paired_explainers_pvt_table)

,pair,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Max.Sens ↓,Mono.Corr ↑,Rand.Logit ↓,Focus
0,FusionPullback - FusionGrad,"0.0070 [-0.0031, 0.0170]","-0.1152 [-0.1330, -0.0974]","-22.5950 [-26.1671, -19.2468]","-0.4421 [-0.4479, -0.4363]","-0.0483 [-0.0627, -0.0338]","-0.0224 [-0.0346, -0.0101]","0.0530 [0.0490, 0.0570]"
1,PullbackAscent - GradientAscent,"0.0618 [0.0505, 0.0734]","0.1115 [0.0842, 0.1392]","-12.9064 [-15.1549, -10.7841]","-0.8275 [-0.8320, -0.8230]","0.0778 [0.0566, 0.0995]","0.1354 [0.1304, 0.1404]","0.0666 [0.0619, 0.0712]"
2,SmoothPullback - SmoothGrad,"0.0322 [0.0216, 0.0431]","-0.0801 [-0.0977, -0.0625]","-38.0143 [-42.3591, -33.8440]","-0.3831 [-0.3867, -0.3794]","-0.0752 [-0.0906, -0.0599]","-0.0237 [-0.0376, -0.0099]","0.0630 [0.0588, 0.0674]"
3,SoftPullback - Gradient,"0.1489 [0.1319, 0.1657]","0.0641 [0.0377, 0.0907]","-83.7863 [-90.4122, -77.2206]","-0.6860 [-0.6922, -0.6799]","0.0589 [0.0364, 0.0815]","-0.0101 [-0.0207, 0.0006]","0.0702 [0.0636, 0.0768]"


,pair,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Max.Sens ↓,Mono.Corr ↑,Rand.Logit ↓,Focus
0,FusionPullback - FusionGrad,"-0.0385 [-0.0537, -0.0233]","-0.0294 [-0.0532, -0.0056]","-48.3830 [-51.9253, -44.9511]","-0.4932 [-0.4997, -0.4867]","-0.0073 [-0.0252, 0.0100]","-0.0372 [-0.0587, -0.0158]","0.0572 [0.0509, 0.0637]"
1,PullbackAscent - GradientAscent,"0.1040 [0.0874, 0.1208]","0.2631 [0.2339, 0.2916]","-24.0957 [-26.3416, -21.9837]","-1.0230 [-1.0278, -1.0182]","0.1903 [0.1655, 0.2149]","0.2105 [0.1999, 0.2210]","0.1419 [0.1353, 0.1487]"
2,SmoothPullback - SmoothGrad,"-0.0058 [-0.0217, 0.0094]","-0.0008 [-0.0259, 0.0239]","-58.7615 [-62.4075, -55.3068]","-0.4187 [-0.4232, -0.4142]","-0.0128 [-0.0314, 0.0053]","-0.0405 [-0.0623, -0.0187]","0.0733 [0.0664, 0.0802]"
3,SoftPullback - Gradient,"0.1472 [0.1226, 0.1710]","0.1475 [0.1170, 0.1783]","-77.3050 [-81.3633, -73.2595]","-0.8335 [-0.8434, -0.8237]","0.1444 [0.1164, 0.1721]","-0.0291 [-0.0509, -0.0067]","0.1116 [0.1023, 0.1208]"


,pair,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Max.Sens ↓,Mono.Corr ↑,Rand.Logit ↓,Focus
0,FusionPullback - FusionGrad,"-0.0088 [-0.0178, 0.0002]","-0.0296 [-0.0486, -0.0106]","-1.5167 [-1.9709, -1.0603]","-0.2258 [-0.2365, -0.2151]","-0.0617 [-0.0774, -0.0462]","0.0061 [-0.0052, 0.0177]","0.0084 [0.0045, 0.0123]"
1,PullbackAscent - GradientAscent,"0.0044 [-0.0109, 0.0199]","0.1415 [0.1196, 0.1642]","-2.8716 [-3.1287, -2.6310]","-0.3875 [-0.3924, -0.3824]","0.0544 [0.0365, 0.0730]","0.1158 [0.1088, 0.1229]","0.0036 [0.0003, 0.0069]"
2,SmoothPullback - SmoothGrad,"0.0059 [-0.0050, 0.0167]","-0.0255 [-0.0461, -0.0050]","-3.8244 [-4.3190, -3.3446]","-0.0497 [-0.0540, -0.0454]","-0.0483 [-0.0641, -0.0324]","0.0200 [0.0055, 0.0345]","0.0411 [0.0363, 0.0459]"
3,SoftPullback - Gradient,"0.0137 [0.0005, 0.0270]","0.0426 [0.0245, 0.0609]","-2.6502 [-3.1834, -2.1285]","0.0317 [0.0190, 0.0439]","-0.0208 [-0.0394, -0.0023]","0.0232 [-0.0030, 0.0496]","0.0599 [0.0541, 0.0655]"


In [27]:
summary_resnet_ablations, csv_resnet_ablations = summarize_ablations_pickle(
    # "ablations 2/quantus_resnet_20_50_ablations_model_name=resnet50_n_batches=50_test_mode=full.pkl"
    "results/ablations/quantus_resnet_20_50_ablations.pkl"
)

Loaded: results\ablations\quantus_resnet_20_50_ablations.pkl
Saved CSV: results\ablations\quantus_resnet_20_50_ablations_summary.csv


,model_name,parameter,value,default_value,explainer,infidelity,faithfulness_correlation,faithfulness_estimate,monotonicity_correlation,max_sensitivity,random_logit
0,resnet50,K,1,5,PullbackAscent,5.955±5.63,0.389±0.37,0.437±0.41,0.418±0.39,0.115±0.03,-0.006±0.43
1,resnet50,K,2,5,PullbackAscent,5.471±5.01,0.396±0.37,0.418±0.42,0.411±0.39,0.139±0.05,0.152±0.31
2,resnet50,K,3,5,PullbackAscent,5.352±4.86,0.393±0.37,0.414±0.42,0.418±0.4,0.179±0.07,0.199±0.26
3,resnet50,K,10,5,PullbackAscent,5.527±5.05,0.358±0.36,0.36±0.45,0.423±0.4,0.358±0.11,0.219±0.14
4,resnet50,alpha,5,20,PullbackAscent,5.393±4.94,0.403±0.37,0.41±0.4,0.418±0.4,0.179±0.08,0.261±0.25
5,resnet50,alpha,10,20,PullbackAscent,5.327±4.85,0.396±0.37,0.405±0.42,0.42±0.4,0.208±0.09,0.261±0.22
6,resnet50,alpha,40,20,PullbackAscent,5.472±5.08,0.362±0.37,0.383±0.45,0.415±0.4,0.305±0.1,0.173±0.19
7,resnet50,default,default,default,PullbackAscent,5.349±4.85,0.382±0.37,0.394±0.43,0.428±0.4,0.245±0.09,0.228±0.2
8,resnet50,default,default,default,SoftPullback,5.974±5.7,0.389±0.37,0.437±0.41,0.419±0.39,0.118±0.04,-0.006±0.43
9,resnet50,tau_maxpool,0.01,0.3,PullbackAscent,5.338±4.86,0.368±0.37,0.331±0.46,0.361±0.39,0.489±0.13,0.2±0.19


In [28]:
summary_resnet_ablations, csv_resnet_ablations = summarize_ablations_pickle(
    # "ablations 2/quantus_pvt_20_50_ablations_model_name=pvt_v2_b1_n_batches=50_test_mode=full.pkl"
    "results/ablations/quantus_pvt_20_50_ablations.pkl"
)

Loaded: results\ablations\quantus_pvt_20_50_ablations.pkl
Saved CSV: results\ablations\quantus_pvt_20_50_ablations_summary.csv


,model_name,parameter,value,default_value,explainer,infidelity,faithfulness_correlation,faithfulness_estimate,monotonicity_correlation,max_sensitivity,random_logit
0,pvt_v2_b1,K,1,5,PullbackAscent,6.224±6.35,0.119±0.34,0.16±0.41,0.164±0.43,1.11±0.16,-0.006±0.35
1,pvt_v2_b1,K,2,5,PullbackAscent,2.686±2.78,0.134±0.35,0.188±0.42,0.215±0.38,1.008±0.13,0.044±0.2
2,pvt_v2_b1,K,3,5,PullbackAscent,1.926±1.64,0.134±0.36,0.202±0.42,0.224±0.35,0.939±0.12,0.073±0.15
3,pvt_v2_b1,K,10,5,PullbackAscent,1.556±0.96,0.103±0.36,0.172±0.43,0.221±0.33,0.8±0.12,0.139±0.07
4,pvt_v2_b1,alpha,5,20,PullbackAscent,1.962±1.66,0.133±0.36,0.155±0.44,0.192±0.38,0.85±0.13,0.111±0.13
5,pvt_v2_b1,alpha,10,20,PullbackAscent,1.702±1.11,0.13±0.36,0.173±0.43,0.209±0.35,0.849±0.13,0.114±0.11
6,pvt_v2_b1,alpha,40,20,PullbackAscent,1.616±1.04,0.11±0.36,0.212±0.42,0.227±0.33,0.867±0.11,0.105±0.1
7,pvt_v2_b1,default,default,default,PullbackAscent,1.619±1.03,0.122±0.36,0.201±0.41,0.229±0.33,0.855±0.12,0.113±0.11
8,pvt_v2_b1,default,default,default,SoftPullback,6.183±6.12,0.119±0.34,0.16±0.41,0.164±0.43,1.065±0.18,-0.006±0.35
9,pvt_v2_b1,tau_attention,0.5,1.0,PullbackAscent,1.681±1.1,0.12±0.36,0.187±0.42,0.22±0.35,0.927±0.11,0.101±0.1


In [29]:
summary_resnet_ablations, csv_resnet_ablations = summarize_ablations_pickle(
    # "ablations 2/quantus_vgg_20_50_ablations_model_name=vgg11_bn_n_batches=50_test_mode=full.pkl"
    "results/ablations/quantus_vgg_20_50_ablations.pkl"
)

Loaded: results\ablations\quantus_vgg_20_50_ablations.pkl
Saved CSV: results\ablations\quantus_vgg_20_50_ablations_summary.csv


,model_name,parameter,value,default_value,explainer,infidelity,faithfulness_correlation,faithfulness_estimate,monotonicity_correlation,max_sensitivity,random_logit
0,vgg11_bn,K,1,5,PullbackAscent,15.933±29.73,0.606±0.3,0.53±0.4,0.366±0.39,0.208±0.07,-0.005±0.32
1,vgg11_bn,K,2,5,PullbackAscent,8.998±20.86,0.601±0.29,0.447±0.45,0.329±0.38,0.251±0.08,0.101±0.21
2,vgg11_bn,K,3,5,PullbackAscent,6.798±17.18,0.594±0.29,0.423±0.46,0.32±0.38,0.304±0.1,0.136±0.18
3,vgg11_bn,K,10,5,PullbackAscent,5.09±8.63,0.537±0.3,0.356±0.47,0.315±0.36,0.521±0.13,0.165±0.1
4,vgg11_bn,alpha,5,20,PullbackAscent,6.321±13.88,0.588±0.29,0.393±0.47,0.321±0.39,0.249±0.09,0.211±0.17
5,vgg11_bn,alpha,10,20,PullbackAscent,5.738±13.28,0.577±0.29,0.372±0.48,0.314±0.38,0.299±0.09,0.206±0.15
6,vgg11_bn,alpha,40,20,PullbackAscent,6.15±15.74,0.57±0.29,0.421±0.45,0.319±0.36,0.538±0.12,0.112±0.13
7,vgg11_bn,default,default,default,PullbackAscent,5.624±13.24,0.57±0.29,0.375±0.48,0.329±0.37,0.389±0.11,0.166±0.14
8,vgg11_bn,default,default,default,SoftPullback,15.702±29.89,0.606±0.3,0.53±0.4,0.369±0.39,0.213±0.08,-0.005±0.32
9,vgg11_bn,tau_maxpool,0.01,0.3,PullbackAscent,4.097±4.51,0.53±0.29,0.296±0.49,0.289±0.33,0.765±0.13,0.133±0.13


In [30]:
paired_df_resnet, paired_pivot_resnet, paired_csv_resnet = paired_differences_vs_default(
    # "results/quantus_ablations_model_name=resnet50_n_batches=25_test_mode=test.pkl"
    "results/ablations/quantus_resnet_20_50_ablations.pkl"
)
# paired_df_pvt, paired_pivot_pvt, paired_csv_pvt = paired_differences_vs_default(
#     "results/quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test.pkl"
# )

Loaded: results\ablations\quantus_resnet_20_50_ablations.pkl
Saved paired difference CSV: results\ablations\quantus_resnet_20_50_ablations_paired_diff_vs_default.csv
Default scores (mean±std):


,model_name,explainer,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Max.Sens ↓,Mono.Corr ↑,Rand.Logit ↓
0,resnet50,PullbackAscent,0.3822±0.3671,0.3943±0.4331,5.3493±4.8502,0.2454±0.0921,0.4282±0.4013,0.2282±0.2019
1,resnet50,SoftPullback,0.3887±0.3735,0.4368±0.4146,5.9736±5.6981,0.1185±0.0365,0.4187±0.3882,-0.0059±0.4345


Loaded: results\ablations\quantus_resnet_20_50_ablations.pkl
Saved paired difference CSV: results\ablations\quantus_resnet_20_50_ablations_paired_diff_vs_default.csv
Default scores (mean±std):


,model_name,explainer,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Max.Sens ↓,Mono.Corr ↑,Rand.Logit ↓
0,resnet50,PullbackAscent,0.3822±0.3671,0.3943±0.4331,5.3493±4.8502,0.2454±0.0921,0.4282±0.4013,0.2282±0.2019
1,resnet50,SoftPullback,0.3887±0.3735,0.4368±0.4146,5.9736±5.6981,0.1185±0.0365,0.4187±0.3882,-0.0059±0.4345


Paired delta format: variant - default, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts present in the detailed CSV: [np.int64(1000)]


metric,explainer,parameter,default_value,value,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Max.Sens ↓,Mono.Corr ↑,Rand.Logit ↓
0,PullbackAscent,K,5.0,1.00,"0.0065 [-0.0011, 0.0142]","0.0429 [0.0290, 0.0567]","0.6062 [0.3976, 0.8236]","-0.1299 [-0.1341, -0.1255]","-0.0100 [-0.0207, 0.0012]","-0.2341 [-0.2505, -0.2176]"
1,PullbackAscent,K,5.0,2.00,"0.0143 [0.0090, 0.0196]","0.0235 [0.0123, 0.0353]","0.1219 [0.0217, 0.2213]","-0.1060 [-0.1093, -0.1028]","-0.0174 [-0.0267, -0.0081]","-0.0762 [-0.0844, -0.0680]"
2,PullbackAscent,K,5.0,3.00,"0.0112 [0.0080, 0.0145]","0.0192 [0.0115, 0.0271]","0.0022 [-0.0548, 0.0572]","-0.0664 [-0.0687, -0.0641]","-0.0102 [-0.0183, -0.0019]","-0.0293 [-0.0338, -0.0249]"
3,PullbackAscent,K,5.0,10.00,"-0.0243 [-0.0288, -0.0198]","-0.0340 [-0.0441, -0.0240]","0.1774 [0.1052, 0.2545]","0.1125 [0.1098, 0.1152]","-0.0056 [-0.0151, 0.0039]","-0.0092 [-0.0137, -0.0049]"
4,PullbackAscent,alpha,20.0,5.00,"0.0206 [0.0157, 0.0255]","0.0155 [0.0053, 0.0258]","0.0441 [-0.0418, 0.1289]","-0.0667 [-0.0695, -0.0639]","-0.0100 [-0.0194, -0.0008]","0.0329 [0.0274, 0.0387]"
5,PullbackAscent,alpha,20.0,10.00,"0.0135 [0.0105, 0.0165]","0.0106 [0.0032, 0.0180]","-0.0224 [-0.0729, 0.0267]","-0.0378 [-0.0399, -0.0356]","-0.0083 [-0.0165, -0.0003]","0.0330 [0.0301, 0.0358]"
6,PullbackAscent,alpha,20.0,40.00,"-0.0201 [-0.0239, -0.0164]","-0.0116 [-0.0209, -0.0029]","0.1227 [0.0481, 0.2049]","0.0595 [0.0574, 0.0617]","-0.0135 [-0.0224, -0.0046]","-0.0555 [-0.0580, -0.0529]"
7,PullbackAscent,tau_maxpool,0.3,0.01,"-0.0139 [-0.0181, -0.0095]","-0.0637 [-0.0771, -0.0507]","-0.0110 [-0.0498, 0.0299]","0.2441 [0.2402, 0.2481]","-0.0675 [-0.0798, -0.0553]","-0.0278 [-0.0303, -0.0253]"
8,PullbackAscent,tau_maxpool,0.3,0.50,"0.0044 [0.0031, 0.0056]","0.0195 [0.0134, 0.0257]","0.0556 [0.0402, 0.0712]","-0.0282 [-0.0294, -0.0271]","0.0026 [-0.0041, 0.0091]","0.0018 [0.0009, 0.0027]"
9,PullbackAscent,tau_relu,0.6,0.30,"-0.0021 [-0.0126, 0.0086]","-0.0534 [-0.0715, -0.0358]","3.0008 [2.4559, 3.6084]","0.3180 [0.3125, 0.3237]","-0.0642 [-0.0795, -0.0486]","-0.1612 [-0.1703, -0.1522]"


In [31]:
paired_df_pvt, paired_pivot_pvt, paired_csv_pvt = paired_differences_vs_default(
    "results/quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test.pkl"
)

Loaded: results\quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test.pkl
Saved paired difference CSV: results\quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test_paired_diff_vs_default.csv
Default scores (mean±std):


,model_name,explainer,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Rand.Logit ↓
0,pvt_v2_b1,PullbackAscent,0.1328±0.3714,0.2015±0.4263,1.6928±1.4162,0.1110±0.1119
1,pvt_v2_b1,SoftPullback,0.0966±0.3304,0.1518±0.3912,6.3481±5.9812,-0.0178±0.3799


Paired delta format: variant - default, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts present in the detailed CSV: [np.int64(500)]


Loaded: results\quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test.pkl
Saved paired difference CSV: results\quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test_paired_diff_vs_default.csv
Default scores (mean±std):


,model_name,explainer,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Rand.Logit ↓
0,pvt_v2_b1,PullbackAscent,0.1328±0.3714,0.2015±0.4263,1.6928±1.4162,0.1110±0.1119
1,pvt_v2_b1,SoftPullback,0.0966±0.3304,0.1518±0.3912,6.3481±5.9812,-0.0178±0.3799


Paired delta format: variant - default, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts present in the detailed CSV: [np.int64(500)]


metric,explainer,parameter,default_value,value,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Rand.Logit ↓
0,PullbackAscent,K,5.0,1.0,"-0.0362 [-0.0651, -0.0066]","-0.0498 [-0.0868, -0.0116]","4.7475 [4.2074, 5.3071]","-0.1288 [-0.1550, -0.1029]"
1,PullbackAscent,K,5.0,2.0,"-0.0085 [-0.0262, 0.0097]","-0.0033 [-0.0296, 0.0232]","1.1071 [0.8526, 1.3782]","-0.0708 [-0.0813, -0.0603]"
2,PullbackAscent,K,5.0,3.0,"0.0010 [-0.0101, 0.0119]","0.0074 [-0.0112, 0.0266]","0.1901 [0.1062, 0.2648]","-0.0405 [-0.0457, -0.0353]"
3,PullbackAscent,K,5.0,10.0,"-0.0126 [-0.0246, -0.0005]","-0.0179 [-0.0389, 0.0033]","-0.1222 [-0.2156, -0.0520]","0.0284 [0.0242, 0.0326]"
4,PullbackAscent,alpha,20.0,5.0,"0.0011 [-0.0121, 0.0148]","-0.0578 [-0.0831, -0.0329]","0.3007 [0.1842, 0.4328]","-0.0035 [-0.0071, 0.0001]"
5,PullbackAscent,alpha,20.0,10.0,"0.0003 [-0.0071, 0.0079]","-0.0251 [-0.0425, -0.0079]","0.0442 [-0.0179, 0.0943]","0.0019 [0.0000, 0.0037]"
6,PullbackAscent,alpha,20.0,40.0,"0.0031 [-0.0067, 0.0127]","0.0326 [0.0141, 0.0515]","-0.0355 [-0.1040, 0.0122]","-0.0068 [-0.0087, -0.0049]"
7,PullbackAscent,tau_attention,1.0,0.5,"-0.0028 [-0.0125, 0.0070]","-0.0065 [-0.0251, 0.0124]","0.0500 [0.0115, 0.0854]","-0.0159 [-0.0206, -0.0114]"
8,PullbackAscent,tau_attention,1.0,5.0,"-0.0149 [-0.0312, 0.0010]","-0.0172 [-0.0391, 0.0054]","-0.0764 [-0.1345, -0.0285]","-0.0014 [-0.0084, 0.0054]"
9,PullbackAscent,tau_gelu,1.0,0.5,"0.0008 [-0.0074, 0.0089]","-0.0197 [-0.0390, -0.0004]","-0.0647 [-0.1492, -0.0032]","-0.0027 [-0.0070, 0.0015]"
